In [ ]:
# ========================================================================
# FAIRFACE 6-WAY SKIN TONE CLASSIFIER
# Using EfficientNet-B4 + L*-based Pseudo-Labels (PyTorch)
# ========================================================================

import os
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast
from torchvision import models, transforms
from PIL import Image

from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, precision_recall_fscore_support,
    precision_recall_curve
)
from sklearn.utils.class_weight import compute_class_weight

# Check device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'  GPU: {torch.cuda.get_device_name(0)}')
    print(f'  Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

In [ ]:
# ========================================================================
# SECTION 1: CONFIGURATION
# ========================================================================

# Paths — update these to match your setup
CSV_PATH = r'datasets/fairface_lstar_labels.csv'
IMAGE_ROOT = r'datasets/fairface-img-margin025-trainval/'
OUTPUT_DIR = r'outputs/fairface/'

# Image settings  (EfficientNet-B4 native: 380x380)
IMAGE_SIZE = 380
BATCH_SIZE = 16
NUM_WORKERS = 4

# Training epochs (2-stage strategy)
EPOCHS_HEAD = 10        # Stage 1: train classifier head only
EPOCHS_FINETUNE = 40    # Stage 2: fine-tune entire network

# Learning rates
LR_HEAD = 1e-3
LR_FINETUNE = 5e-6

# Regularization
DROPOUT_RATE = 0.5

# Focal loss parameters
FOCAL_GAMMA = 2.0
FOCAL_ALPHA = 0.25

# Early stopping
PATIENCE = 15

# Number of classes
NUM_CLASSES = 6

# Class names (ordered by class index 0-5)
CLASS_NAMES = [
    'Type VI (darkest)',   # class 0
    'Type V',              # class 1
    'Type IV',             # class 2
    'Type III',            # class 3
    'Type II',             # class 4
    'Type I (lightest)',   # class 5
]

os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"\n{'='*60}")
print(f"CONFIGURATION: 6-WAY FAIRFACE SKIN TONE CLASSIFICATION")
print(f"{'='*60}")
print(f"  Model:      EfficientNet-B4")
print(f"  Image size: {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs:     {EPOCHS_HEAD} (head) + {EPOCHS_FINETUNE} (fine-tune)")
print(f"  Device:     {device}")

In [ ]:
# ========================================================================
# SECTION 2: DATA LOADING & SPLIT
# ========================================================================

print("\n--- Loading FairFace L* labels ---")
df = pd.read_csv(CSV_PATH)
print(f"Total images: {len(df):,}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nSample rows:")
display(df.head())

# ========================================================================
# CLASS DISTRIBUTION
# ========================================================================

print("\nClass distribution:")
dist = df['skin_tone_class'].value_counts().sort_index(ascending=False)
for cls_id in sorted(dist.index, reverse=True):
    count = dist[cls_id]
    pct = count / len(df) * 100
    print(f"  Class {cls_id} ({CLASS_NAMES[cls_id]:20s}): {count:6,}  ({pct:5.1f}%)")

# ========================================================================
# TRAIN / VALIDATION SPLIT (uses FairFace original splits)
# ========================================================================

print("\n--- Using FairFace original train/val split ---")
train_df = df[df['original_split'] == 'train'].reset_index(drop=True)
val_df = df[df['original_split'] == 'val'].reset_index(drop=True)

print(f"  Train: {len(train_df):,} images")
print(f"  Val:   {len(val_df):,} images")

# ========================================================================
# COMPUTE CLASS WEIGHTS FOR IMBALANCED DATA
# ========================================================================

print("\n--- Computing class weights for imbalanced data ---")
train_labels = train_df['skin_tone_class'].values
unique_classes = np.unique(train_labels)
class_weights_array = compute_class_weight('balanced', classes=unique_classes, y=train_labels)

# Build full weight vector (even if some classes are missing)
full_weights = np.ones(NUM_CLASSES, dtype=np.float32)
for cls, w in zip(unique_classes, class_weights_array):
    full_weights[cls] = w

class_weights_tensor = torch.tensor(full_weights, dtype=torch.float32).to(device)

print("\nClass weights:")
for i, name in enumerate(CLASS_NAMES):
    count = (train_labels == i).sum()
    print(f"  {name:20s}: {full_weights[i]:.3f}  (n={count:,})")

In [ ]:
# ========================================================================
# SECTION 3: DATA PIPELINE (SKIN-TONE FOCUSED AUGMENTATION)
# ========================================================================

print(f"\n--- Building PyTorch data pipelines (IMAGE_SIZE={IMAGE_SIZE}) ---")

# Training transforms — mirrors skin_tone_augment() from train_fitzpatrick.py
train_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ColorJitter(
        brightness=0.15,
        contrast=0.2,
        saturation=0.2,
        hue=0.05,
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])

# Validation transforms — deterministic resize + center crop
val_transforms = transforms.Compose([
    transforms.Resize(IMAGE_SIZE + 32),
    transforms.CenterCrop(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225],
    ),
])


class FairFaceDataset(Dataset):
    """PyTorch Dataset for FairFace images with L*-based skin tone labels."""

    def __init__(self, dataframe, image_root, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_root = image_root
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = os.path.join(self.image_root, row['file'])

        try:
            image = Image.open(img_path).convert('RGB')
        except (FileNotFoundError, OSError):
            warnings.warn(f'Could not load image: {img_path}')
            image = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), (128, 128, 128))

        if self.transform:
            image = self.transform(image)

        label = int(row['skin_tone_class'])
        return image, label


# Build datasets and dataloaders
train_dataset = FairFaceDataset(train_df, IMAGE_ROOT, transform=train_transforms)
val_dataset = FairFaceDataset(val_df, IMAGE_ROOT, transform=val_transforms)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=NUM_WORKERS, pin_memory=True, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=NUM_WORKERS, pin_memory=True,
)

print(f"  Train batches: {len(train_loader)}")
print(f"  Val batches:   {len(val_loader)}")

# ========================================================================
# FOCAL LOSS IMPLEMENTATION
# ========================================================================

class FocalLoss(nn.Module):
    """
    Multi-class Focal Loss (Lin et al., 2017).
    Reduces loss for well-classified examples; focuses on hard ones.
    """
    def __init__(self, gamma=2.0, alpha=0.25, weight=None):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha
        self.weight = weight

    def forward(self, logits, targets):
        ce_loss = nn.functional.cross_entropy(
            logits, targets, weight=self.weight, reduction='none'
        )
        pt = torch.exp(-ce_loss)
        focal_weight = self.alpha * (1 - pt) ** self.gamma
        return (focal_weight * ce_loss).mean()

criterion = FocalLoss(gamma=FOCAL_GAMMA, alpha=FOCAL_ALPHA, weight=class_weights_tensor)

print(f"\n--- Focal Loss configured ---")
print(f"  Gamma: {FOCAL_GAMMA} (focus on hard examples)")
print(f"  Alpha: {FOCAL_ALPHA} (balancing parameter)")

In [ ]:
# ========================================================================
# SECTION 4: BUILDING THE EFFICIENTNET-B4 MODEL
# ========================================================================

print(f"\n--- Building transfer learning model with EfficientNet-B4 ---")

model = models.efficientnet_b4(weights=models.EfficientNet_B4_Weights.IMAGENET1K_V1)

# Replace classifier head
in_features = model.classifier[1].in_features   # 1792 for B4
model.classifier = nn.Sequential(
    nn.Dropout(p=DROPOUT_RATE),
    nn.Linear(in_features, NUM_CLASSES),
)

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
print(f"  Total parameters:     {total_params:,}")
print(f"  Classifier input dim: {in_features}")
print(f"  Output classes:       {NUM_CLASSES}")

In [ ]:
# ========================================================================
# SECTION 5: STAGE 1 — TRAINING HEAD WITH FOCAL LOSS & CLASS WEIGHTS
# ========================================================================

print(f"\n--- STAGE 1: Training the classification head ---")

# Freeze backbone
for name, param in model.named_parameters():
    if 'classifier' not in name:
        param.requires_grad = False

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")

optimizer = optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=LR_HEAD,
)
scaler = GradScaler(enabled=(device.type == 'cuda'))

# Training history
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_val_acc = 0.0

for epoch in range(EPOCHS_HEAD):
    t0 = time.time()

    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS_HEAD}', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validate ---
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    dt = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_head_model.pth'))
        improved = ' ★ saved'

    print(
        f"  Epoch {epoch+1:02d}/{EPOCHS_HEAD}  "
        f"loss={train_loss:.4f}  acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
        f"({dt:.1f}s){improved}"
    )

# Load best head model
model.load_state_dict(
    torch.load(os.path.join(OUTPUT_DIR, 'best_head_model.pth'),
               map_location=device, weights_only=True)
)
print("✓ Best head model loaded")

In [ ]:
# ========================================================================
# SECTION 6: STAGE 2 — FINE-TUNING WITH LOWER LEARNING RATE
# ========================================================================

print(f"\n--- STAGE 2: Fine-tuning the entire model ---")

# Unfreeze all layers
for param in model.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"  Trainable parameters: {trainable:,}")

# Mark stage boundary for plotting
history['stage_boundary'] = EPOCHS_HEAD

optimizer_ft = optim.AdamW(model.parameters(), lr=LR_FINETUNE, weight_decay=1e-5)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer_ft, T_0=10, T_mult=2, eta_min=1e-7,
)

epochs_no_improve = 0

for epoch in range(EPOCHS_FINETUNE):
    global_epoch = EPOCHS_HEAD + epoch
    t0 = time.time()

    # --- Train ---
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in tqdm(train_loader, desc=f'Epoch {global_epoch+1}/{EPOCHS_HEAD+EPOCHS_FINETUNE}', leave=False):
        images, labels = images.to(device), labels.to(device)
        optimizer_ft.zero_grad()

        with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer_ft)
        scaler.update()

        running_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = correct / total

    # --- Validate ---
    model.eval()
    val_loss_sum, val_correct, val_total = 0.0, 0, 0

    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
                outputs = model(images)
                loss = criterion(outputs, labels)
            val_loss_sum += loss.item() * images.size(0)
            _, preds = outputs.max(1)
            val_correct += preds.eq(labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total
    scheduler.step()
    dt = time.time() - t0

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    improved = ''
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'best_finetuned_model.pth'))
        improved = ' ★ saved'
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    print(
        f"  Epoch {global_epoch+1:02d}/{EPOCHS_HEAD+EPOCHS_FINETUNE}  "
        f"loss={train_loss:.4f}  acc={train_acc:.4f}  "
        f"val_loss={val_loss:.4f}  val_acc={val_acc:.4f}  "
        f"lr={optimizer_ft.param_groups[0]['lr']:.2e}  "
        f"({dt:.1f}s){improved}"
    )

    # Early stopping
    if epochs_no_improve >= PATIENCE:
        print(f"\n  ⚠ Early stopping at epoch {global_epoch+1} "
              f"(no improvement for {PATIENCE} epochs)")
        break

# Save final model
torch.save(model.state_dict(), os.path.join(OUTPUT_DIR, 'final_model.pth'))
print(f"\n✓ Final model saved to {os.path.join(OUTPUT_DIR, 'final_model.pth')}")

# Load best fine-tuned model for evaluation
best_path = os.path.join(OUTPUT_DIR, 'best_finetuned_model.pth')
if os.path.exists(best_path):
    model.load_state_dict(
        torch.load(best_path, map_location=device, weights_only=True)
    )
print("✓ Best fine-tuned model loaded")

In [ ]:
# ========================================================================
# SECTION 7: ENHANCED EVALUATION WITH ADVANCED METRICS
# ========================================================================

# --- 1. Plot training history ---

def plot_training_history(history):
    """Plot training curves (loss + accuracy)."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history['train_loss'], label='Train Loss', linewidth=2)
    ax1.plot(history['val_loss'], label='Val Loss', linewidth=2)
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training & Validation Loss')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history['train_acc'], label='Train Acc', linewidth=2)
    ax2.plot(history['val_acc'], label='Val Acc', linewidth=2)
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training & Validation Accuracy')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    if 'stage_boundary' in history:
        for ax in (ax1, ax2):
            ax.axvline(x=history['stage_boundary'], color='red',
                       linestyle='--', alpha=0.5, label='Fine-tune start')
            ax.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fairface_training_curves.png'), dpi=150)
    plt.show()

print("\n--- Plotting combined training history ---")
plot_training_history(history)


# --- 2. Collect predictions on validation set ---

@torch.no_grad()
def collect_predictions(model, loader):
    """Collect all predictions and labels from a data loader."""
    model.eval()
    all_preds = []
    all_probs = []
    all_labels = []

    for images, labels in tqdm(loader, desc='Predicting'):
        images = images.to(device)
        with autocast(device_type='cuda', enabled=(device.type == 'cuda')):
            outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        _, preds = probs.max(1)

        all_preds.extend(preds.cpu().numpy())
        all_probs.append(probs.cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.array(all_preds), np.vstack(all_probs), np.array(all_labels)

print("\n--- Collecting validation predictions ---")
val_preds, val_probs, val_labels = collect_predictions(model, val_loader)


# --- 3. Classification report ---

print(f"\n{'='*60}")
print("CLASSIFICATION REPORT")
print(f"{'='*60}")
print(classification_report(
    val_labels, val_preds,
    target_names=CLASS_NAMES,
    labels=range(NUM_CLASSES),
    zero_division=0,
    digits=3,
))

acc = accuracy_score(val_labels, val_preds)
prec, rec, f1, _ = precision_recall_fscore_support(
    val_labels, val_preds, average='macro', zero_division=0
)
print(f"  Macro Accuracy:  {acc:.4f}")
print(f"  Macro Precision: {prec:.4f}")
print(f"  Macro Recall:    {rec:.4f}")
print(f"  Macro F1:        {f1:.4f}")


# --- 4. Confusion matrix ---

print(f"\n--- Confusion Matrix ---")
cm = confusion_matrix(val_labels, val_preds, labels=range(NUM_CLASSES))
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax,
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix — FairFace Skin Tone Classifier', fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fairface_confusion_matrix.png'), dpi=150)
plt.show()


# --- 5. Precision-Recall curves ---

print(f"\n--- Precision-Recall Curves ---")
n_cols = 3
n_rows = (NUM_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(5*n_cols, 4*n_rows))
axes = axes.flatten()

for i, class_name in enumerate(CLASS_NAMES):
    y_true_binary = (val_labels == i).astype(int)
    precision_vals, recall_vals, _ = precision_recall_curve(y_true_binary, val_probs[:, i])

    axes[i].plot(recall_vals, precision_vals, linewidth=2)
    axes[i].set_xlabel('Recall', fontsize=10)
    axes[i].set_ylabel('Precision', fontsize=10)
    axes[i].set_title(f'PR: {class_name}', fontsize=11, fontweight='bold')
    axes[i].grid(True, alpha=0.3)
    axes[i].set_xlim([0, 1])
    axes[i].set_ylim([0, 1])

    support = y_true_binary.sum() / len(y_true_binary)
    axes[i].axhline(y=support, color='r', linestyle='--', alpha=0.5,
                    label=f'Baseline ({support:.2f})')
    axes[i].legend(loc='lower left', fontsize=9)

for i in range(NUM_CLASSES, len(axes)):
    axes[i].axis('off')

plt.tight_layout()
plt.show()


# --- 6. Error analysis ---

print(f"\n--- Error Analysis ---")
error_mask = val_preds != val_labels
n_errors = error_mask.sum()
print(f"Total errors: {n_errors} out of {len(val_labels)} ({n_errors/len(val_labels)*100:.1f}%)")

print("\nErrors by true class:")
for i, class_name in enumerate(CLASS_NAMES):
    class_mask = val_labels == i
    class_total = class_mask.sum()
    class_errors = (error_mask & class_mask).sum()
    if class_total > 0:
        error_rate = class_errors / class_total * 100
        print(f"  {class_name:20s}: {class_errors:3d}/{class_total:3d} ({error_rate:5.1f}% error rate)")


# --- 7. Final summary ---

print(f"\n{'='*60}")
print(f"FINAL RESULTS — 6-WAY FAIRFACE CLASSIFICATION")
print(f"{'='*60}")
print(f"  Model:                   EfficientNet-B4")
print(f"  Best Validation Accuracy: {best_val_acc:.4f}")
print(f"  Validation Accuracy:      {acc:.4f}")
print(f"  Macro F1:                 {f1:.4f}")
print(f"  Models saved to:          {OUTPUT_DIR}")
print(f"{'='*60}")